In [1]:
# ==============================================================
# Hindi Morphological Tagger — UD Hindi HDTB only
# ==============================================================

import os, sys, subprocess, warnings, zipfile, urllib.request
from pathlib import Path
from collections import Counter

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"]   = "900"
os.environ["HF_HUB_ETAG_TIMEOUT"]       = "60"
os.environ["HF_HUB_DISABLE_TELEMETRY"]  = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

def _pip(*pkgs):
    for p in pkgs:
        try:
            __import__(p.split("[")[0].replace("-", "_"))
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])

_pip("numpy", "pandas", "scikit-learn", "openpyxl")
_pip("torch", "datasets", "transformers")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
from sklearn.metrics import classification_report

warnings.filterwarnings("ignore")
print("✅ Dependencies ready.")

OUTPUT_DIR = Path(r"D:\DATASET On 10-12-2025\Objective 1\OB1.1\Dataset\Morph_Tagger_Report")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
UD_CACHE = OUTPUT_DIR / "ud_hindi_hdtb"

# ==============================================================
# 1. LOAD UD HINDI HDTB
# ==============================================================
print("\n[1/5] Loading UD Hindi HDTB ...")

def _download_with_retries(url, dest, max_retries=5):
    for attempt in range(1, max_retries + 1):
        try:
            print(f"   Attempt {attempt}/{max_retries} ...")
            with urllib.request.urlopen(url, timeout=120) as r, open(dest, "wb") as f:
                while True:
                    chunk = r.read(1 << 20)
                    if not chunk:
                        break
                    f.write(chunk)
            return True
        except Exception as e:
            print(f"   ⚠ {e}")
    return False

def parse_conllu(path):
    sentences, sent = [], []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line or line.startswith("#"):
                if sent:
                    sentences.append(sent)
                    sent = []
                continue
            parts = line.split("\t")
            if len(parts) >= 10 and parts[0].isdigit():
                feats = {}
                if parts[5] != "_":
                    for feat in parts[5].split("|"):
                        if "=" in feat:
                            k, v = feat.split("=", 1)
                            feats[k] = v
                sent.append({"form": parts[1], "upos": parts[3], "feats": feats})
    if sent:
        sentences.append(sent)
    return sentences

def normalize_split(sentences):
    return [{
        "tokens": [w["form"] for w in s],
        "upos":   [w["upos"] for w in s],
        "feats":  [w["feats"] for w in s]
    } for s in sentences]

ud_train = ud_dev = ud_test = None

try:
    from datasets import load_dataset
    ud = load_dataset("universal_dependencies", "hi_hdtb")
    ud_train = [{"tokens": ex["tokens"], "upos": ex["upos"], "feats": ex["feats"]} for ex in ud["train"]]
    ud_dev   = [{"tokens": ex["tokens"], "upos": ex["upos"], "feats": ex["feats"]} for ex in ud["validation"]]
    ud_test  = [{"tokens": ex["tokens"], "upos": ex["upos"], "feats": ex["feats"]} for ex in ud["test"]]
    print(f"   ✓ UD HDTB: train={len(ud_train)}, dev={len(ud_dev)}, test={len(ud_test)}")
except Exception:
    print("   ⚠ datasets library failed; using GitHub zip ...")
    UD_CACHE.mkdir(parents=True, exist_ok=True)
    zip_path = UD_CACHE / "ud_hindi_hdtb.zip"
    base_ok = UD_CACHE / "UD_Hindi-HDTB-master" / "hi_hdtb-ud-train.conllu"
    if not base_ok.exists():
        if not zip_path.exists():
            ok = _download_with_retries(
                "https://github.com/UniversalDependencies/UD_Hindi-HDTB/archive/refs/heads/master.zip",
                zip_path
            )
            if not ok:
                raise RuntimeError("Could not download UD Hindi HDTB.")
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(UD_CACHE)
    base = UD_CACHE / "UD_Hindi-HDTB-master"
    ud_train = normalize_split(parse_conllu(base / "hi_hdtb-ud-train.conllu"))
    ud_dev   = normalize_split(parse_conllu(base / "hi_hdtb-ud-dev.conllu"))
    ud_test  = normalize_split(parse_conllu(base / "hi_hdtb-ud-test.conllu"))
    print(f"   ✓ UD HDTB: train={len(ud_train)}, dev={len(ud_dev)}, test={len(ud_test)}")

# ==============================================================
# 2. BUILD LABEL SETS
# ==============================================================
print("\n[2/5] Building label sets ...")

def build_label_set(data, task):
    labels = set()
    for sent in data:
        for feat in sent["feats"]:
            if task in feat:
                labels.add(feat[task])
        if task == "upos":
            labels.update(sent["upos"])
    return sorted(labels)

label_maps = {
    "upos":   build_label_set(ud_train, "upos"),
    "Gender": build_label_set(ud_train, "Gender"),
    "Number": build_label_set(ud_train, "Number"),
    "Case":   build_label_set(ud_train, "Case"),
}
for task in ["Gender", "Number", "Case"]:
    if "O" not in label_maps[task]:
        label_maps[task].append("O")

label2id = {task: {l: i for i, l in enumerate(labels)} for task, labels in label_maps.items()}
id2label = {task: {i: l for l, i in m.items()} for task, m in label2id.items()}

print(f"   POS: {len(label_maps['upos'])} labels")
print(f"   Gender: {label_maps['Gender']}")
print(f"   Number: {label_maps['Number']}")
print(f"   Case: {label_maps['Case']}")

# ==============================================================
# 3. CLASS WEIGHTS
# ==============================================================
print("\n[3/5] Computing class weights ...")

def compute_class_weights(data, task, id2label_map):
    counts = Counter()
    n_classes = len(id2label_map)
    for sent in data:
        if task == "upos":
            for u in sent["upos"]:
                counts[u] += 1
        else:
            for feat in sent["feats"]:
                counts[feat.get(task, "O")] += 1
    total = sum(counts.values())
    weights = []
    for i in range(n_classes):
        label = id2label_map[i]
        c = counts.get(label, 1)
        weights.append(total / (n_classes * max(c, 1)))
    return torch.tensor(weights, dtype=torch.float)

class_weights = {
    "upos":   compute_class_weights(ud_train, "upos",   id2label["upos"]),
    "Gender": compute_class_weights(ud_train, "Gender", id2label["Gender"]),
    "Number": compute_class_weights(ud_train, "Number", id2label["Number"]),
    "Case":   compute_class_weights(ud_train, "Case",   id2label["Case"]),
}

# ==============================================================
# 4. MULTI-TASK MODEL
# ==============================================================
print("\n[4/5] Defining multi-task model ...")

class MultiTaskMorphTagger(nn.Module):
    def __init__(self, model_name, label_maps, class_weights):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.config = self.encoder.config
        hidden_size = self.config.hidden_size

        self.pos_head    = nn.Linear(hidden_size, len(label_maps["upos"]))
        self.gender_head = nn.Linear(hidden_size, len(label_maps["Gender"]))
        self.number_head = nn.Linear(hidden_size, len(label_maps["Number"]))
        self.case_head   = nn.Linear(hidden_size, len(label_maps["Case"]))

        total_feat_dim = (
            len(label_maps["upos"]) + len(label_maps["Gender"]) +
            len(label_maps["Number"]) + len(label_maps["Case"])
        )
        self.morph_proj = nn.Linear(total_feat_dim, 128)

        self.register_buffer("w_pos",    class_weights["upos"])
        self.register_buffer("w_gender", class_weights["Gender"])
        self.register_buffer("w_number", class_weights["Number"])
        self.register_buffer("w_case",   class_weights["Case"])

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        seq = outputs.last_hidden_state

        pos_logits    = self.pos_head(seq)
        gender_logits = self.gender_head(seq)
        number_logits = self.number_head(seq)
        case_logits   = self.case_head(seq)

        pos_probs    = torch.softmax(pos_logits, dim=-1)
        gender_probs = torch.softmax(gender_logits, dim=-1)
        number_probs = torch.softmax(number_logits, dim=-1)
        case_probs   = torch.softmax(case_logits, dim=-1)

        combined = torch.cat([pos_probs, gender_probs, number_probs, case_probs], dim=-1)
        morph_embedding = self.morph_proj(combined)

        loss = None
        if labels is not None:
            loss_fct_w_pos    = nn.CrossEntropyLoss(ignore_index=-100, weight=self.w_pos)
            loss_fct_w_gender = nn.CrossEntropyLoss(ignore_index=-100, weight=self.w_gender)
            loss_fct_w_number = nn.CrossEntropyLoss(ignore_index=-100, weight=self.w_number)
            loss_fct_w_case   = nn.CrossEntropyLoss(ignore_index=-100, weight=self.w_case)

            pos_loss    = loss_fct_w_pos(pos_logits.view(-1, pos_logits.size(-1)),    labels["upos"].view(-1))
            gender_loss = loss_fct_w_gender(gender_logits.view(-1, gender_logits.size(-1)), labels["Gender"].view(-1))
            number_loss = loss_fct_w_number(number_logits.view(-1, number_logits.size(-1)), labels["Number"].view(-1))
            case_loss   = loss_fct_w_case(case_logits.view(-1, case_logits.size(-1)),   labels["Case"].view(-1))
            loss = pos_loss + gender_loss + number_loss + case_loss

        return {
            "loss": loss,
            "pos_logits": pos_logits,
            "gender_logits": gender_logits,
            "number_logits": number_logits,
            "case_logits": case_logits,
            "morph_embedding": morph_embedding,
        }

class MorphDataset(torch.utils.data.Dataset):
    def __init__(self, data, tokenizer, label2id, max_len=128):
        self.data, self.tok, self.l2i, self.max_len = data, tokenizer, label2id, max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ex = self.data[idx]
        enc = self.tok(
            ex["tokens"],
            is_split_into_words=True,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        word_ids = enc.word_ids()
        labels = {task: [] for task in ["upos", "Gender", "Number", "Case"]}
        for task in labels:
            prev = None
            for wid in word_ids:
                if wid is None or wid != prev:
                    if task == "upos":
                        labels[task].append(self.l2i["upos"][ex["upos"][wid]] if wid is not None else -100)
                    else:
                        feat_val = ex["feats"][wid].get(task, "O") if wid is not None else "O"
                        labels[task].append(self.l2i[task][feat_val] if wid is not None else -100)
                else:
                    labels[task].append(-100)
                prev = wid
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": {task: torch.tensor(v) for task, v in labels.items()},
        }

tokenizer, model, model_used = None, None, None
for candidate in ["google/muril-base-cased", "distilbert-base-multilingual-cased"]:
    try:
        print(f"   Trying: {candidate} ...")
        tok = AutoTokenizer.from_pretrained(candidate, use_fast=True)
        mdl = MultiTaskMorphTagger(candidate, label_maps, class_weights)
        tokenizer, model, model_used = tok, mdl, candidate
        print(f"   ✓ Loaded: {candidate}")
        break
    except Exception as e:
        print(f"   ⚠ Failed: {candidate} — {e}")

if model is None:
    raise RuntimeError("Could not load any model.")

# ==============================================================
# 5. TRAIN
# ==============================================================
print("\n[5/5] Training multi-task model ...")

train_ds = MorphDataset(ud_train, tokenizer, label2id)
val_ds   = MorphDataset(ud_dev,   tokenizer, label2id)
test_ds  = MorphDataset(ud_test,  tokenizer, label2id)

def custom_collate(batch):
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "labels": {
            task: torch.stack([b["labels"][task] for b in batch])
            for task in ["upos", "Gender", "Number", "Case"]
        },
    }

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "mtl_tagger"),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    report_to="none",
    seed=42,
    fp16=torch.cuda.is_available(),
    save_strategy="no",
    disable_tqdm=True,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=custom_collate,
)
trainer.train()

# ==============================================================
# 6. EVALUATE
# ==============================================================
print("\nEvaluating on held-out UD Hindi test split ...")
model.eval()
device = next(model.parameters()).device

metrics = {task: {"y_true": [], "y_pred": []} for task in ["upos", "Gender", "Number", "Case"]}

with torch.no_grad():
    for batch in torch.utils.data.DataLoader(test_ds, batch_size=16, collate_fn=custom_collate):
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        out  = model(ids, mask)

        for task, key in [
            ("upos", "pos_logits"),
            ("Gender", "gender_logits"),
            ("Number", "number_logits"),
            ("Case", "case_logits"),
        ]:
            preds = torch.argmax(out[key], dim=-1).cpu().numpy()
            true  = batch["labels"][task].numpy()
            for p, t in zip(preds, true):
                for pi, ti in zip(p, t):
                    if ti != -100:
                        metrics[task]["y_true"].append(id2label[task][ti])
                        metrics[task]["y_pred"].append(id2label[task][pi])

reports = {}
for task in metrics:
    report = classification_report(
        metrics[task]["y_true"],
        metrics[task]["y_pred"],
        digits=4,
        zero_division=0
    )
    reports[task] = report
    print(f"\n=== {task} Report ===")
    print(report)

# ==============================================================
# 7. WRITE REPORT
# ==============================================================
report_path = OUTPUT_DIR / "MORPH_TAGGER_REPORT_UD_ONLY.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("# Morphological Tagger: UD Hindi HDTB Only\n\n")
    f.write("## 1. Source\n")
    f.write(f"- Model: `{model_used}` fine-tuned with four task heads.\n")
    f.write("- 128-dim morphology embedding via learned projection.\n\n")

    f.write("## 2. Training Data\n")
    f.write(f"- **UD Hindi HDTB:** {len(ud_train)} train sentences.\n")
    f.write(f"- **Dev:** {len(ud_dev)} sentences.\n")
    f.write(f"- **Test:** {len(ud_test)} sentences.\n")
    f.write("- LDC2010T24 and IndicCorpus were not used.\n\n")

    f.write("## 3. Inference-Time Procedure\n")
    f.write(
        "At each decoding step, the tagger receives the caption prefix, predicts "
        "POS/Gender/Number/Case for the next token, and produces a 128-dim embedding "
        "`m_i` fed to MAA. No gold tags are required at test time.\n\n"
    )

    f.write("## 4. Independent Evaluation (UD Hindi test split)\n")
    for task, report in reports.items():
        f.write(f"### {task}\n```\n{report}\n```\n\n")

print(f"\n✅ Report written to: {report_path}")

✅ Dependencies ready.

[1/5] Loading UD Hindi HDTB ...
   ⚠ datasets library failed; using GitHub zip ...
   ✓ UD HDTB: train=13306, dev=1659, test=1684

[2/5] Building label sets ...
   POS: 16 labels
   Gender: ['Fem', 'Masc', 'O']
   Number: ['Plur', 'Sing', 'O']
   Case: ['Acc', 'Acc,Dat', 'Acc,Erg', 'Acc,Gen', 'Acc,Ine', 'Acc,Ins', 'Nom', 'O']

[3/5] Computing class weights ...

[4/5] Defining multi-task model ...
   Trying: google/muril-base-cased ...
   ✓ Loaded: google/muril-base-cased

[5/5] Training multi-task model ...
{'loss': 6.9404, 'grad_norm': 0.9813874363899231, 'learning_rate': 1.920673076923077e-05, 'epoch': 0.1201923076923077}
{'loss': 6.5983, 'grad_norm': 1.4389060735702515, 'learning_rate': 1.840544871794872e-05, 'epoch': 0.2403846153846154}
{'loss': 6.3866, 'grad_norm': 1.5459035634994507, 'learning_rate': 1.760416666666667e-05, 'epoch': 0.3605769230769231}
{'loss': 6.2162, 'grad_norm': 1.6403793096542358, 'learning_rate': 1.6802884615384616e-05, 'epoch': 0.48076

In [2]:
# ==============================================================
# 6b. INFERENCE-TIME MORPHOLOGY (no gold tags)
# ==============================================================
print("\nDemonstrating inference-time morphology (no gold tags) ...")

model.eval()
device = next(model.parameters()).device

def morphology_for_prefix(prefix_tokens):
    """
    Given the Hindi caption prefix generated so far (list of tokens),
    predict UPOS/Gender/Number/Case and return the 128-dim morph embedding m_t.
    No gold tags are used.
    """
    enc = tokenizer(
        prefix_tokens,
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        out = model(enc["input_ids"], enc["attention_mask"])

    # Take the last real token position, not the padding
    last_idx = enc["attention_mask"].sum(dim=1) - 1          # [1]
    pos_logits    = out["pos_logits"][0, last_idx, :]         # [16]
    gender_logits = out["gender_logits"][0, last_idx, :]      # [3]
    number_logits = out["number_logits"][0, last_idx, :]      # [3]
    case_logits   = out["case_logits"][0, last_idx, :]        # [C]

    p_pos    = torch.softmax(pos_logits,    dim=-1)
    p_gender = torch.softmax(gender_logits, dim=-1)
    p_number = torch.softmax(number_logits, dim=-1)
    p_case   = torch.softmax(case_logits,   dim=-1)

    combined = torch.cat([p_pos, p_gender, p_number, p_case], dim=-1)  # [1, F]
    m_t = model.morph_proj(combined)                                   # [1, 128]

    return {
        "upos":        id2label["upos"][int(torch.argmax(p_pos))],
        "Gender":      id2label["Gender"][int(torch.argmax(p_gender))],
        "Number":      id2label["Number"][int(torch.argmax(p_number))],
        "Case":        id2label["Case"][int(torch.argmax(p_case))],
        "m_t_shape":   tuple(m_t.shape),
        "m_t_norm":    float(m_t.norm()),
    }

# Example: two prefixes of a caption being generated
for prefix in [
    ["एक", "महिला"],
    ["एक", "महिला", "पार्क", "में"],
]:
    print(f"\nPrefix: {' '.join(prefix)}")
    print("  ", morphology_for_prefix(prefix))

print("\nNote: no gold tags were used at inference.")


Demonstrating inference-time morphology (no gold tags) ...

Prefix: एक महिला
   {'upos': 'ADP', 'Gender': 'O', 'Number': 'O', 'Case': 'O', 'm_t_shape': (1, 128), 'm_t_norm': 1.641461968421936}

Prefix: एक महिला पार्क में
   {'upos': 'PUNCT', 'Gender': 'O', 'Number': 'O', 'Case': 'O', 'm_t_shape': (1, 128), 'm_t_norm': 1.641563057899475}

Note: no gold tags were used at inference.
